# 稀疏态演化示例

本教程展示 SparseState 如何随算子操作演化。

## 初始状态

SparseState 默认创建单个 |0...0⟩ 基态。

In [ ]:
import pysparq as ps

ps.System.clear()

# 创建 2 比特寄存器
ps.System.add_register("q", ps.UnsignedInteger, 2)

# 初始状态
state = ps.SparseState()

print("初始状态:")
ps.StatePrint()(state)
print(f"\n基态数量: {state.size()}")

## Hadamard 变换

Hadamard 创建量子叠加态。

In [ ]:
# Hadamard_Int: 对指定数量的量子比特应用 Hadamard
ps.Hadamard_Int("q", 1)(state)

print("Hadamard_Int(q, 1) 后:")
ps.StatePrint()(state)
print(f"\n基态数量: {state.size()}")

In [ ]:
# Hadamard_Int_Full: 对所有量子比特应用完整 Hadamard
ps.Hadamard_Int_Full("q")(state)

print("Hadamard_Int_Full(q) 后:")
ps.StatePrint()(state)
print(f"\n基态数量: {state.size()}")

## 状态打印模式

StatePrint 支持多种显示模式。

In [ ]:
print("Default 模式:")
ps.StatePrint(ps.Default)(state)

In [ ]:
print("Binary 模式:")
ps.StatePrint(ps.Binary)(state)

In [ ]:
print("Prob 模式:")
ps.StatePrint(ps.Prob)(state)

## 算术操作演化

In [ ]:
ps.System.clear()

# 创建寄存器
ps.System.add_register("a", ps.UnsignedInteger, 2)
ps.System.add_register("b", ps.UnsignedInteger, 2)
ps.System.add_register("sum", ps.UnsignedInteger, 2)

state = ps.SparseState()

# 初始化
ps.Init_Unsafe("a", 1)(state)
ps.Init_Unsafe("b", 2)(state)

print("初始状态:")
ps.StatePrint()(state)

In [ ]:
# 加法: sum ^= a + b
ps.Add_UInt_UInt("a", "b", "sum")(state)

print("Add_UInt_UInt 后:")
ps.StatePrint()(state)
# sum = 0 ^ (1 + 2) = 3

In [ ]:
# 再次应用 = 撤销（XOR 机制）
ps.Add_UInt_UInt("a", "b", "sum")(state)

print("再次 Add_UInt_UInt 后:")
ps.StatePrint()(state)
# sum = 3 ^ 3 = 0

## 叠加态上的算术

In [ ]:
ps.System.clear()

ps.System.add_register("x", ps.UnsignedInteger, 2)
ps.System.add_register("y", ps.UnsignedInteger, 2)

state = ps.SparseState()

# x 处于叠加态
ps.Hadamard_Int_Full("x")(state)

# y 初始化为常数
ps.Init_Unsafe("y", 1)(state)

print("初始叠加态:")
ps.StatePrint()(state)

In [ ]:
# 内置加法: y += x（每个分支独立计算）
ps.Add_UInt_UInt_InPlace("x", "y")(state)

print("Add_UInt_UInt_InPlace 后:")
ps.StatePrint()(state)
# 每个分支: y = 1 + x

In [ ]:
# 撤销
ps.Add_UInt_UInt_InPlace("x", "y").dag(state)

print("dagger 后:")
ps.StatePrint()(state)

## 访问基态数据

In [ ]:
# 遍历所有基态
print("遍历基态:")
for i, system in enumerate(state.basis_states):
    x_id = ps.System.get_id("x")
    y_id = ps.System.get_id("y")
    
    x_val = system.get(x_id).value
    y_val = system.get(y_id).value
    amp = system.amplitude
    
    print(f"  基态 {i}: x={x_val}, y={y_val}, 振幅={amp}")

## 总结

- SparseState 只存储非零基态
- Hadamard 创建叠加态，基态数量增加
- 算术操作作用于每个基态
- SelfAdjointOperator 两次应用恢复原值
- BaseOperator 使用 dag() 撤销